In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive


In [ ]:

df1 = pd.read_excel("/content/clean_dataset_1_agnews.xlsx")
df1 = df1[['text', 'label']]


labels_list1 = df1['label'].unique().tolist()
label2id1 = {label: i for i, label in enumerate(labels_list1)}
df1['label'] = df1['label'].map(label2id1)

hf_dataset1 = Dataset.from_pandas(df1)
dataset_split1 = hf_dataset1.train_test_split(test_size=0.2, seed=42)
train_dataset1 = dataset_split1['train']
eval_dataset1 = dataset_split1['test']
num_labels1 = len(labels_list1)



df2 = pd.read_excel("/content/clean_dataset_2_legaldispute.xlsx")
df2 = df2[['text', 'label']]


labels_list2 = df2['label'].unique().tolist()
label2id2 = {label: i for i, label in enumerate(labels_list2)}
df2['label'] = df2['label'].map(label2id2)

hf_dataset2 = Dataset.from_pandas(df2)
dataset_split2 = hf_dataset2.train_test_split(test_size=0.2, seed=42)
train_dataset2 = dataset_split2['train']
eval_dataset2 = dataset_split2['test']
num_labels2 = len(labels_list2)

print(f"Dataset 1 - Train: {len(train_dataset1)}, Eval: {len(eval_dataset1)}")
print(f"Dataset 1 Labels Mapping: {label2id1}")

print(f"\nDataset 2 - Train: {len(train_dataset2)}, Eval: {len(eval_dataset2)}")
print(f"Dataset 2 Labels Mapping: {label2id2}")

Dataset 1 - Train: 1600, Eval: 400
Dataset 1 Labels Mapping: {'Business': 0, 'SciTech': 1, 'Sport': 2, 'World': 3}

Dataset 2 - Train: 1600, Eval: 400
Dataset 2 Labels Mapping: {'02_quality_dispute': 0, '03_no_contract': 1, '01_delayed_payment': 2, '04_partial_payment': 3}


In [ ]:
from transformers import DistilBertTokenizer



tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):

    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)



tokenized_train1 = train_dataset1.map(tokenize_function, batched=True)

tokenized_eval1 = eval_dataset1.map(tokenize_function, batched=True)



tokenized_train2 = train_dataset2.map(tokenize_function, batched=True)

tokenized_eval2 = eval_dataset2.map(tokenize_function, batched=True)

print("DistilBERT Tokenization Complete!")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

DistilBERT Tokenization Complete!


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    acc = accuracy_score(labels, predictions)

    return {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

In [ ]:
import math
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification

weight_decays = [0.01, 0.1]
batch_sizes = [16, 32]
learning_rates = [0.00002, 0.00003]

results_list = []

for wd in weight_decays:
    for bs in batch_sizes:
        for lr in learning_rates:
            print(f"\n=======================================================")
            print(f"Training Combination: LR={lr}, Batch={bs}, WD={wd} (DistilBERT - 1 Epoch)")
            print(f"=======================================================\n")


            print(">>> Training on Dataset 1...")


            model1 = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels1)

            total_steps1 = math.ceil(len(tokenized_train1) / bs) * 1
            warmup_steps1 = int(total_steps1 * 0.1)

            output_dir1 = f'./DistilBERT_Models/D1_lr{lr}_bs{bs}_wd{wd}'
            training_args1 = TrainingArguments(
                output_dir=output_dir1,
                num_train_epochs=1,
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                weight_decay=wd,
                warmup_steps=warmup_steps1,
                eval_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="F1"
            )

            trainer1 = Trainer(
                model=model1, args=training_args1,
                train_dataset=tokenized_train1, eval_dataset=tokenized_eval1,
                compute_metrics=compute_metrics
            )
            trainer1.train()
            eval_results1 = trainer1.evaluate()

            trainer1.save_model(f'{output_dir1}/best_model')


            print("\n>>> Training on Dataset 2...")


            model2 = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels2)

            total_steps2 = math.ceil(len(tokenized_train2) / bs) * 1
            warmup_steps2 = int(total_steps2 * 0.1)

            output_dir2 = f'./DistilBERT_Models/D2_lr{lr}_bs{bs}_wd{wd}'
            training_args2 = TrainingArguments(
                output_dir=output_dir2,
                num_train_epochs=1,
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                weight_decay=wd,
                warmup_steps=warmup_steps2,
                eval_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="F1"
            )

            trainer2 = Trainer(
                model=model2, args=training_args2,
                train_dataset=tokenized_train2, eval_dataset=tokenized_eval2,
                compute_metrics=compute_metrics
            )
            trainer2.train()
            eval_results2 = trainer2.evaluate()

            trainer2.save_model(f'{output_dir2}/best_model')

            results_list.append({
                'Model': 'DistilBERT',
                'Learning Rate': lr,
                'Batch Size': bs,
                'Weight Decay': wd,
                'Dataset 1 Acc': eval_results1['eval_Accuracy'],
                'Dataset 1 Prec': eval_results1['eval_Precision'],
                'Dataset 1 Rec': eval_results1['eval_Recall'],
                'Dataset 1 F1': eval_results1['eval_F1'],
                'Dataset 2 Acc': eval_results2['eval_Accuracy'],
                'Dataset 2 Prec': eval_results2['eval_Precision'],
                'Dataset 2 Rec': eval_results2['eval_Recall'],
                'Dataset 2 F1': eval_results2['eval_F1']
            })


Training Combination: LR=2e-05, Batch=16, WD=0.01 (DistilBERT - 1 Epoch)

>>> Training on Dataset 1...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.287858,0.437500,0.266146,0.437500,0.329054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,1.287858,1,0.437500,0.266146,0.437500,0.329054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:


results_df = pd.DataFrame(results_list)

print("\n=======================================================")
print("FINAL COMBINED RESULTS TABLE")
print("=======================================================\n")

display(results_df)



file_name = "bert_experiments_combined_results.xlsx"

results_df.to_excel(
    file_name,
    index=False,
    engine="openpyxl"
)

print(
    f"\nফলাফল সফলভাবে '{file_name}' ফাইলে সেভ করা হয়েছে।"
)



print(
    f"\nTotal Hyperparameter Combinations Completed: {len(results_df)}"
)


FINAL COMBINED RESULTS TABLE



,Model,Learning Rate,Batch Size,Weight Decay,Dataset 1 Acc,Dataset 1 Prec,Dataset 1 Rec,Dataset 1 F1,Dataset 2 Acc,Dataset 2 Prec,Dataset 2 Rec,Dataset 2 F1
0,DistilBERT,0.00002,16,0.01,0.800,0.792695,0.800,0.792458,0.4775,0.413435,0.4775,0.370269
1,DistilBERT,0.00003,16,0.01,0.830,0.842891,0.830,0.831491,0.5350,0.617120,0.5350,0.485178
2,DistilBERT,0.00002,32,0.01,0.750,0.758287,0.750,0.724469,0.3725,0.234372,0.3725,0.237309
3,DistilBERT,0.00003,32,0.01,0.810,0.806216,0.810,0.803366,0.4325,0.262695,0.4325,0.324727
4,DistilBERT,0.00002,16,0.10,0.835,0.834948,0.835,0.832925,0.4775,0.413001,0.4775,0.370072
5,DistilBERT,0.00003,16,0.10,0.830,0.842891,0.830,0.831491,0.5300,0.607358,0.5300,0.482590
6,DistilBERT,0.00002,32,0.10,0.750,0.758287,0.750,0.724469,0.3750,0.238496,0.3750,0.238683
7,DistilBERT,0.00003,32,0.10,0.810,0.806216,0.810,0.803366,0.4375,0.266146,0.4375,0.329054



ফলাফল সফলভাবে 'bert_experiments_combined_results.xlsx' ফাইলে সেভ করা হয়েছে।

Total Hyperparameter Combinations Completed: 8
